# Audio Flamingo 3 (NVIDIA)

In [ ]:
import torch
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
from llm_utils.config import PROJECT_ROOT, CACHE_DIR
MODEL_ID     = "nvidia/audio-flamingo-3-hf"


processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=CACHE_DIR,
)

print("Audio Flamingo 3 model loaded.")

Loading weights: 100%|██████████| 830/830 [00:18<00:00, 44.15it/s] 


Audio Flamingo 3 model loaded.


In [ ]:
PAIR_NUM = 1

SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="flamingo3_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [ ]:
def classify_audio(wav_path: Path, control_wavs: list, dementia_wavs: list) -> str:
    """Few-shot classify a single audio file. Returns raw model response."""
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    ]
    for wav in control_wavs:
        conversation.append({
            "role": "user",
            "content": [
                {"type": "text",  "text": USER_PROMPT + "\n\nThis is a healthy control speaker:"},
                {"type": "audio", "path": str(wav)},
            ],
        })
        conversation.append({
            "role": "assistant",
            "content": [{"type": "text", "text": "Control"}],
        })
    for wav in dementia_wavs:
        conversation.append({
            "role": "user",
            "content": [
                {"type": "text",  "text": USER_PROMPT + "\n\nThis is a dementia speaker:"},
                {"type": "audio", "path": str(wav)},
            ],
        })
        conversation.append({
            "role": "assistant",
            "content": [{"type": "text", "text": "Dementia"}],
        })
    conversation.append({
        "role": "user",
        "content": [
            {"type": "text",  "text": USER_PROMPT},
            {"type": "audio", "path": str(wav_path)},
        ],
    })

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=64)
    output = processor.batch_decode(
        generated_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return output[0]


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [ ]:
OUTPUT_DIR = Path("/root/autodl-tmp/Back_to_Origin/LLM/results/audio_flamingo3_fewshot_result")


def get_examples(df, audio_dir, n=PAIR_NUM):
    """Pick the first n Control and first n Dementia samples as few-shot examples."""
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    examples = {}
    for ad_val, label in label_map.items():
        rows = df[df["ad"] == ad_val].iloc[:n]
        wavs, ids = [], []
        for _, row in rows.iterrows():
            matches = list(audio_dir.glob(f"{label}/{row['session_id']}.*"))
            wavs.append(ensure_wav(matches[0]))
            ids.append(row["session_id"])
        examples[label] = {"session_ids": ids, "wavs": wavs}
    print(f"  Few-shot examples: Control={examples['Control']['session_ids']}, Dementia={examples['Dementia']['session_ids']}")
    return examples


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    examples = get_examples(df, audio_dir)
    example_ids = set(examples["Control"]["session_ids"] + examples["Dementia"]["session_ids"])
    control_wavs  = examples["Control"]["wavs"]
    dementia_wavs = examples["Dementia"]["wavs"]

    predictions, skipped = [], 0

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        if row["session_id"] in example_ids:
            skipped += 1
            continue
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]), control_wavs, dementia_wavs)
            pred = parse_prediction(raw)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            raw, pred = "OOM", None
        except Exception as e:
            raw, pred = str(e), None
        finally:
            torch.cuda.empty_cache()
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1) * 100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) * 100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [8]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt-origin", "Pitt-origin", "Pitt-origin_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt-origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

[Pitt-origin-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/Pitt-origin, exists=True


Pitt-origin-raw:   0%|          | 1/552 [00:02<19:41,  2.14s/it]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-raw:   0%|          | 2/552 [00:03<13:46,  1.50s/it]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-raw:   1%|          | 3/552 [00:04<11:34,  1.27s/it]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-raw: 100%|██████████| 552/552 [08:43<00:00,  1.05it/s]

[Pitt-origin-raw]
  Accuracy:    0.5598
  F1:          0.7178
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 552/552  Skipped: 0


In [9]:
# csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)

# audio_dir = PROJECT_ROOT / "data/raw/Pitt"
# evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [10]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS", "ADReSS", "ADReSS_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS"
evaluate_dataset(csv, audio_dir, "ADReSS-raw")

[ADReSS-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/ADReSS, exists=True


ADReSS-raw:   1%|          | 1/156 [00:00<02:34,  1.00it/s]

  DEBUG [0] session=S001 raw='Dementia' pred=Dementia


ADReSS-raw:   1%|▏         | 2/156 [00:01<02:04,  1.24it/s]

  DEBUG [1] session=S002 raw='Dementia' pred=Dementia


ADReSS-raw:   2%|▏         | 3/156 [00:02<02:03,  1.24it/s]

  DEBUG [2] session=S003 raw='Dementia' pred=Dementia


ADReSS-raw: 100%|██████████| 156/156 [02:21<00:00,  1.10it/s]

[ADReSS-raw]
  Accuracy:    0.5000
  F1:          0.6667
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 156/156  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/ADReSSo-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSSo", "ADReSSo", "ADReSSo_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSSo"
evaluate_dataset(csv, audio_dir, "ADReSSo-raw")

[ADReSSo-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/ADReSSo, exists=True


ADReSSo-raw:   0%|          | 1/237 [00:00<02:19,  1.69it/s]

  DEBUG [0] session=adrsdt10 raw='Dementia' pred=Dementia


ADReSSo-raw:   1%|          | 2/237 [00:01<02:54,  1.34it/s]

  DEBUG [1] session=adrsdt11 raw='Dementia' pred=Dementia


ADReSSo-raw:   1%|▏         | 3/237 [00:02<03:05,  1.26it/s]

  DEBUG [2] session=adrsdt12 raw='Dementia' pred=Dementia


ADReSSo-raw: 100%|██████████| 237/237 [03:40<00:00,  1.07it/s]

[ADReSSo-raw]
  Accuracy:    0.5148
  F1:          0.6797
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 237/237  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-M-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS-M", "ADReSS-M", "ADReSS-M_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS-M"
evaluate_dataset(csv, audio_dir, "ADReSS-M-raw")

[ADReSS-M-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/ADReSS-M, exists=True


ADReSS-M-raw:   0%|          | 1/237 [00:01<03:59,  1.01s/it]

  DEBUG [0] session=adrso002 raw='Dementia' pred=Dementia


ADReSS-M-raw:   1%|          | 2/237 [00:01<02:51,  1.37it/s]

  DEBUG [1] session=adrso003 raw='Dementia' pred=Dementia


ADReSS-M-raw:   1%|▏         | 3/237 [00:01<02:12,  1.77it/s]

  DEBUG [2] session=adrso004 raw='Dementia' pred=Dementia


ADReSS-M-raw: 100%|██████████| 237/237 [04:06<00:00,  1.04s/it]

[ADReSS-M-raw]
  Accuracy:    0.5148
  F1:          0.6797
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 237/237  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:00<00:52,  1.40it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:01<00:45,  1.57it/s]

  DEBUG [1] session=F22_001 raw="Dementia'" pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:02<00:47,  1.48it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [00:45<00:00,  1.62it/s]

[Lu-raw]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0
